# 군집 진단 — 경계 혼동은 "같은 뜻 조항이 다른 문서에서 다른 라벨을 받은 자리"

E5 임베딩을 **분류 특징으로 쓰는 것은 v5에서도 실패했다**(decisions-09 18:40,
`cluster_universality.py`). 이 노트북은 같은 임베딩을 점수가 아니라 **진단**에 쓴다.
어느 셀도 모델을 학습하지 않는다.

묻는 것은 하나다. 모델이 틀리는 자리에서, **뜻이 비슷한 조항은 다른 RFP에서 어떤
라벨을 받았나.** 비슷한 문장이 다른 문서에서 대부분 같은 라벨이면 모델이 못 배운 것이고,
대부분 다른 라벨이면 판정이 문장이 아니라 문서 맥락에서 온 것이다.

노트북 20은 같은 질문을 v5·v6 불일치로 물었지만, v6 프롬프트가 경계 정의를 직접 바꿔
순환이 있었다(decisions-08 08:34 정정). 여기서는 **라벨 하나(v5)와 임베딩만** 쓴다.

로직은 [`scripts/evaluation/cluster_diagnostics.py`](../scripts/evaluation/cluster_diagnostics.py)에
있고 노트북은 그것을 부른다. 실행 전 `RFP_DATASET_VERSION=v5`.


In [ ]:
"""준비 — v5 라벨, E5 임베딩(캐시), 기준선 OOF, v6 라벨, 앵커."""
import collections
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from scripts.evaluation import cluster_diagnostics as D
from scripts.labeling.label_dataset import load_label_dataset

rows, meta = load_label_dataset()
version = meta["dataset_version"].rsplit("_", 1)[-1]
assert version == "v5", f"RFP_DATASET_VERSION=v5 로 실행한다 (지금 {version})"
uid = [r["requirement_uid"] for r in rows]
doc = np.array([r["document_id"] for r in rows])
lab = np.array([r["primary_action"] for r in rows])
oof, v6, anchors = D.load_oof(version), D.load_v6(), D.load_anchor_uids()
groups = D.group_masks(rows, oof, v6)

emb = D.e5_matrix(rows, version)
sim_e = emb @ emb.T
sim_t = D.tfidf_similarity(rows)          # 대조군: 기준선과 같은 word+char TF-IDF
print(f"{version} {len(rows)}건 · 평가(앵커 제외) {int(groups['모델이 맞힌 건'].sum() + groups['경계 밖 오답'].sum() + groups['경계 혼동'].sum())}건")
print({k: int(v.sum()) for k, v in groups.items() if k in ("모델이 맞힌 건", "경계 밖 오답", "경계 혼동", "v5·v6 불일치")})


In [ ]:
"""핵심 — 다른 문서 최근접 5건의 다수결 라벨이 자기 라벨과 맞는 비율.

같은 문서 안의 이웃은 뺀다. 같은 문서는 같은 사업·같은 라벨러 맥락이라 "다른 RFP에서도
그런가"를 묻지 못하기 때문이다. E5와 TF-IDF 두 표현으로 재서 나란히 놓는다.
"""
nn_e, nn_sim_e, major_e, agree_e = D.cross_document_neighbors(sim_e, doc, lab)
nn_t, _, major_t, agree_t = D.cross_document_neighbors(sim_t, doc, lab)
overlap = np.mean([len(set(a) & set(b)) / D.K_NEIGHBORS for a, b in zip(nn_e, nn_t)])

table_e, table_t = D.consistency_table(agree_e, groups), D.consistency_table(agree_t, groups)
frame = pd.DataFrame({
    "건수": {k: v["n"] for k, v in table_e.items()},
    "E5 이웃 일치": {k: f"{v['rate']*100:.1f}%" for k, v in table_e.items()},
    "E5 Wilson 95%": {k: f"{v['wilson95'][0]*100:.1f}~{v['wilson95'][1]*100:.1f}" for k, v in table_e.items()},
    "TF-IDF 이웃 일치": {k: f"{v['rate']*100:.1f}%" for k, v in table_t.items()},
    "TF-IDF Wilson 95%": {k: f"{v['wilson95'][0]*100:.1f}~{v['wilson95'][1]*100:.1f}" for k, v in table_t.items()},
})
display(frame)
print(f"두 표현이 고른 이웃 5건의 겹침 평균 {overlap*100:.1f}% — 다른 이웃을 골라도 비율이 같다.")
from sklearn.metrics import f1_score
print(f"이 다수결을 그대로 예측으로 쓰면 macro F1  E5 {f1_score(lab, major_e, labels=D.LABELS, average='macro'):.3f} / "
      f"TF-IDF {f1_score(lab, major_t, labels=D.LABELS, average='macro'):.3f}  (학습 없음, 기준선 0.640)")


In [ ]:
"""숫자보다 사례 — 경계 혼동 중 이웃 5건이 전부 다른 라벨인 건을 읽는다.

문장 대부분은 다른 RFP의 견적 항목과 같은데, 이 문서에만 있는 조건 하나가 라벨을 바꾼다.
"""
pred = {u: oof[u]["word_char_logistic_pred"] for u in uid if u in oof}
boundary_idx = np.where(groups["경계 혼동"])[0]
zero_agree = [i for i in boundary_idx if (lab[nn_e[i]] == lab[i]).sum() == 0]
print(f"경계 혼동 {len(boundary_idx)}건 중 이웃 5건이 전부 다른 라벨인 건 {len(zero_agree)}건\n")

shown = 0
for i in sorted(zero_agree, key=lambda j: len(rows[j]["raw_requirement_text"])):
    if len(rows[i]["raw_requirement_text"]) < 120:
        continue
    r = rows[i]
    print(f"■ [{uid[i]}] {r['requirement_name'][:40]}")
    print(f"   v5 라벨 {lab[i]}  /  모델 {pred[uid[i]]}  /  v6 {v6.get(uid[i], '—')}")
    print("   " + re.sub(r"\s+", " ", r["raw_requirement_text"])[:220])
    for j, s in zip(nn_e[i], nn_sim_e[i]):
        print(f"     이웃 {s:.2f}  {doc[j]:<30} {rows[j]['requirement_name'][:20]:<20} → {lab[j]}")
    print()
    shown += 1
    if shown == 3:
        break


In [ ]:
"""덤 둘 — 아웃라이어는 오답과 무관한가, 앵커는 의미 공간을 얼마나 덮나."""
mask, nearest, cut = D.outliers(sim_e)
wrong = groups["경계 밖 오답"] | groups["경계 혼동"]
evaluated = wrong | groups["모델이 맞힌 건"]
hangul = np.array([len(re.findall(r"[가-힣]", r["raw_requirement_text"])) / max(1, len(r["raw_requirement_text"])) for r in rows])
length = np.array([len(r["raw_requirement_text"]) for r in rows])
print(f"― 아웃라이어 (최근접 유사도 하위 5%, 임계 {cut:.3f}, {int(mask.sum())}건) " + "―" * 12)
print(f"  모델 오답률   아웃라이어 {wrong[mask].sum()/evaluated[mask].sum()*100:.1f}%  /  나머지 {wrong[~mask].sum()/evaluated[~mask].sum()*100:.1f}%   → 같다. 오답의 원인이 아니다")
print(f"  한글 비율     아웃라이어 {np.median(hangul[mask]):.2f}  /  나머지 {np.median(hangul[~mask]):.2f}   → 같다. 추출 결함이 아니다")
print(f"  원문 길이     아웃라이어 {np.median(length[mask]):.0f}자  /  나머지 {np.median(length[~mask]):.0f}자   → 짧고 구체적인 항목일 뿐이다")
print("  문서별:", dict(collections.Counter(doc[mask].tolist()).most_common(6)))

cl, covered, uncovered, mixed = D.anchor_coverage(emb, rows, anchors)
print(f"\n― 앵커 {len(anchors)}건이 덮는 군집 (KMeans k={D.K_CLUSTERS}) " + "―" * 12)
print(f"  앵커가 있는 군집 {len(covered)}/{D.K_CLUSTERS}   앵커 없는 군집의 요구사항 {int(uncovered.sum())}건 ({uncovered.mean()*100:.1f}%)")
sizes = collections.Counter(cl.tolist())
for size, c in sorted(((sizes[c], c) for c in range(D.K_CLUSTERS) if c not in covered), reverse=True)[:5]:
    names = collections.Counter(rows[i]["requirement_name"] for i in np.where(cl == c)[0]).most_common(3)
    print(f"    군집 {c:2d} ({size}건, 앵커 0): " + ", ".join(f"{k}×{v}" for k, v in names))
print(f"  다수 라벨 점유율 60% 미만인 군집의 요구사항 {mixed}건 ({mixed/len(rows)*100:.1f}%)")
print("  issues/008의 '인출 유사도 중앙값 0.090'은 이 빈 군집에서 나온다. 라벨 1:1:1로 뽑으면 의미 공간의 40%가 비어도 알 수 없다.")


## 결론

**경계 혼동 127건은 뜻이 비슷한 조항이 다른 RFP에서 대부분 다른 라벨을 받은 자리다.**

| 집단 | E5 이웃 일치 | TF-IDF 이웃 일치 |
|---|---:|---:|
| 모델이 맞힌 건 (944) | 78.9% | 82.8% |
| 경계 밖 오답 (274) | 30.3% | 29.9% |
| **경계 혼동 (127)** | **26.0%** | **22.8%** |

구간이 겹치지 않고, 이웃을 38%만 공유하는 두 표현이 같은 비율을 낸다. 표현이 아니라
데이터의 성질이다. 사례를 읽으면 구조가 보인다 — 국방 RFP의 "AI 플랫폼 SW 도입"은 다른
네 기관의 같은 조항이 전부 `견적반영`인데, **국방망**이라는 한 단어 때문에
`계약·질의검토`다. 문장 대부분은 다른 RFP의 견적 항목과 같고, 이 문서에만 통하는 조건
하나가 라벨을 바꾼다.

그래서 이 127건은 모델이 못 배운 것이 아니라 **배울 표본이 없는** 것이다. 비슷한 문장이
다른 곳에서는 반대로 라벨돼 있으니 어떤 텍스트 모델이든 여기서는 틀린다. 2026-09-02의
28건 정독이 낸 "판정에 문서 밖 지식이 필요하다"를 전수에서 숫자로 확인했다.

세 가지를 함께 적는다.

- **아웃라이어는 오답과 무관하다.** 오답률·한글 비율이 나머지와 같다. 추출 결함도 아니다.
- **앵커 100건은 군집 80개 중 48개만 덮는다.** 요구사항 28.6%가 앵커 없는 군집에 있다.
  앵커 풀은 동결이라 고치지 않으며, 재설계 시 층화 축을 군집으로 잡으라는 기록으로 남긴다.
- **학습 없는 문서 간 kNN이 macro F1 0.60**이다. 기준선 0.640, 앙상블 0.67~0.68이 그 위에
  얹는 것은 0.04~0.08뿐이다. "다른 문서에서 비슷한 조항이 받은 라벨"이 과제의 대부분이다.

기록: decisions-09 2026-09-07 19:20, 19:50. 재현: `reports/current/v5/cluster_diagnostics.json`.
